In [ ]:
# Example 1: public API and check_version window semantics
# The version windows follow pyvers implement_for.check_version (pyvers/implement_for.py, MIT).
import importlib.metadata

import pyvers
from pyvers import implement_for

print("pyvers dist version:", importlib.metadata.version("pyvers"))
print("public API:", pyvers.__all__)

# from_version is inclusive, to_version is exclusive
assert implement_for.check_version("2.1.0", "2.0.0", None)
assert not implement_for.check_version("1.26.4", "2.0.0", None)
assert not implement_for.check_version("2.0.0", None, "2.0.0")
assert implement_for.check_version("1.9.9", None, "2.0.0")
assert implement_for.check_version("3.0.0", None, None)
print("check_version window semantics OK (inclusive from, exclusive to)")

In [ ]:
# Example 2: version-gated dispatch — the README's headline pattern.
# Adapted from the pyvers README (MIT).
from pyvers import get_backend, implement_for, register_backend
import numpy as np

register_backend(group="numpy", backends={"numpy": "numpy"})

@implement_for("numpy")
def create_mask(arr):
    raise NotImplementedError("No matching numpy version found")

@create_mask.register(from_version=None, to_version="2.0.0")
def _(arr):
    np_ = get_backend("numpy")
    return np_.array([x > 0 for x in arr], dtype=np_.bool8)

@create_mask.register(from_version="2.0.0")
def _(arr):
    np_ = get_backend("numpy")
    return np_.array([x > 0 for x in arr], dtype=np_.bool_)

mask = create_mask([-1, 2, -3, 4])
assert mask.tolist() == [False, True, False, True]
print("dispatched against numpy", np.__version__, "->", mask.dtype)

In [ ]:
# Example 3: dispatch agrees with check_version for the installed numpy
from pyvers import implement_for
import numpy as np

@implement_for("numpy")
def numpy_era():
    raise NotImplementedError("no matching numpy version registered")

@numpy_era.register(from_version=None, to_version="2.0.0")
def _():
    return "numpy 1.x era"

@numpy_era.register(from_version="2.0.0")
def _():
    return "numpy 2.x era"

era = numpy_era()
expected = "numpy 2.x era" if implement_for.check_version(np.__version__, "2.0.0", None) else "numpy 1.x era"
assert era == expected, (era, expected, np.__version__)
print(f"numpy {np.__version__} dispatched to: {era}")

In [ ]:
# Example 4: backend registry (register_backend, get_backend, set_backend)
import json

from pyvers import get_backend, register_backend, set_backend

register_backend(group="demo", backends={"demo_np": "numpy", "demo_json": "json"})
print("default backend:", get_backend("demo").__name__)

with set_backend("demo", "json"):
    assert get_backend("demo") is json
    print("inside set_backend ctx:", get_backend("demo").__name__)
print("after ctx:", get_backend("demo").__name__)

register_backend(group="np", backends={"np_alias": "numpy"})
lin = get_backend("np", submodule="linalg")
assert lin.__name__ == "numpy.linalg"
print("submodule fetch through backend:", lin.__name__)

try:
    register_backend(group="other", backends={"np_alias": "numpy"})
    raise AssertionError("backend-name reuse across groups must raise")
except ValueError as e:
    print("cross-group name reuse rejected:", e)

In [ ]:
# Example 5: dispatch with a missing module
from pyvers import implement_for

@implement_for("a_module_that_is_not_installed_xyz")
def needs_missing_module():
    return "should never run"

try:
    needs_missing_module()
    raise AssertionError("expected a dispatch failure for a missing module")
except ModuleNotFoundError as e:
    print("raised as documented:", type(e).__name__, "-", e)